# 04 — Model Comparison Analysis

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("../..").resolve()))
sys.path.insert(0, str(Path(".").resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.evaluation import ResultsManager

RESULTS_DIR  = Path("results")
FEATURE_MODE = "features"         # features | ohlcv
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

rm  = ResultsManager(RESULTS_DIR)
_df = rm.load_all_results()
df  = _df[_df["feature_mode"] == FEATURE_MODE].copy()
print(f"feature_mode filter: {FEATURE_MODE!r}")
print(f"Total experiments loaded: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head()


In [ ]:
# Normalização de colunas entre pipelines ML e DL.
# Cria colunas unificadas para comparação justa:
#   oos_f1_macro, oos_sharpe, oos_mcc, oos_mdd, oos_f1_macro_std, category
#
# Rastreamento de prefixos (load_all_results adiciona ml_/fin_ ao flat dict):
#   ML  ml_metrics  → ml_<key>       (ex: ml_f1_macro)
#   ML  fin_metrics → fin_<key>      (ex: fin_oos_sharpe)
#   DL  ml_metrics  → ml_<key>       (IS fold avg, ex: ml_f1_macro)
#   DL  fin_metrics → fin_<key>      (IS: fin_sharpe; OOS ML: fin_oos_f1_macro; OOS fin: fin_oos_sharpe)

ML_MODELS = ["RandomForest", "XGBoost", "LightGBM", "CatBoost", "Stacking"]
DL_MODELS = ["CNN", "LSTM", "CNN_LSTM", "MLP", "Transformer"]

def _coalesce(row, *cols):
    for c in cols:
        v = row.get(c, float("nan"))
        if pd.notna(v):
            return v
    return float("nan")

# F1-macro OOS:  ML=ml_f1_macro  |  DL=fin_oos_f1_macro
df["oos_f1_macro"] = df.apply(lambda r: _coalesce(r, "fin_oos_f1_macro", "ml_f1_macro"), axis=1)

# Sharpe OOS: both ML and DL store as fin_oos_sharpe
df["oos_sharpe"] = df.apply(lambda r: _coalesce(r, "fin_oos_sharpe"), axis=1)

# MCC OOS:  ML=ml_mcc  |  DL=fin_oos_mcc
df["oos_mcc"] = df.apply(lambda r: _coalesce(r, "fin_oos_mcc", "ml_mcc"), axis=1)

# Max Drawdown OOS: both ML and DL store as fin_oos_max_drawdown
df["oos_mdd"] = df.apply(lambda r: _coalesce(r, "fin_oos_max_drawdown"), axis=1)

# F1-macro std (estabilidade entre janelas): ML=ml_f1_macro_std  |  DL=fin_oos_f1_macro_std
df["oos_f1_macro_std"] = df.apply(lambda r: _coalesce(r, "fin_oos_f1_macro_std", "ml_f1_macro_std"), axis=1)

# Category: ML ou arquitetura DL
df["category"] = df["model_name"].apply(lambda m: "ML" if m in ML_MODELS else m)

print("Colunas unificadas criadas:")
for col in ["oos_f1_macro", "oos_sharpe", "oos_mcc", "oos_mdd", "oos_f1_macro_std"]:
    nn = df[col].notna().sum()
    print(f"  {col:<24} não-NaN: {nn}/{len(df)}")
df[["ticker","model_name","mode","category","oos_f1_macro","oos_sharpe"]].head(8)


In [ ]:
print("Experiments by mode:")
print(df["mode"].value_counts())
print("\nExperiments by model:")
print(df["model_name"].value_counts())
print("\nExperiments by ticker:")
print(df["ticker"].value_counts().to_string())

## 1. Overall Rankings — F1-macro

In [ ]:
rank = (
    df.groupby(["model_name", "mode"])["oos_f1_macro"]
    .agg(["mean", "std", "count"])
    .round(4)
    .sort_values("mean", ascending=False)
)
rank.columns = ["F1-macro OOS mean", "F1-macro OOS std", "n_experiments"]
print("=== Model × Mode — F1-macro OOS (mean across all tickers) ===")
print(rank.to_string())

## 2. Wavelet Impact: raw vs db4 vs learned_wavelet

In [ ]:
dl_df = df[df["mode"].isin(["raw", "db4", "learned_wavelet", "learned_wavelet_no_warmup"])].copy()
fig, axes = plt.subplots(1, 4, figsize=(18, 5), sharey=True)
models = ["CNN", "LSTM", "CNN_LSTM", "Transformer"]
for ax, model in zip(axes, models):
    sub = dl_df[dl_df["model_name"] == model]
    if sub.empty:
        continue
    order = ["raw", "db4", "learned_wavelet", "learned_wavelet_no_warmup"]
    order = [o for o in order if o in sub["mode"].unique()]
    sns.boxplot(data=sub, x="mode", y="oos_f1_macro", order=order, ax=ax,
                palette=["#95a5a6", "#3498db", "#e74c3c", "#f39c12"][:len(order)])
    ax.set_title(model, fontsize=11)
    ax.set_xlabel("")
    ax.set_ylabel("F1-macro OOS" if ax == axes[0] else "")
    ax.tick_params(axis="x", rotation=30)
plt.suptitle("F1-macro OOS by Mode — raw vs db4 vs Learned Wavelet", fontsize=12)
plt.tight_layout()
plt.show()

## 3. Best Model per Ticker

In [ ]:
best_per_ticker = (
    df.loc[df.groupby("ticker")["oos_f1_macro"].idxmax()]
    [["ticker", "model_name", "mode", "oos_f1_macro", "oos_sharpe"]]
    .set_index("ticker")
    .sort_values("oos_f1_macro", ascending=False)
)
print("Best model per ticker (by OOS F1-macro):")
print(best_per_ticker.round(4).to_string())
print(f"\nMost common best model: {best_per_ticker['model_name'].mode().iloc[0]}")
print(f"Most common best mode:  {best_per_ticker['mode'].mode().iloc[0]}")

## 4. Heatmap — F1-macro by Ticker × Model

In [ ]:
pivot = df.pivot_table(
    index="ticker", columns="model_name", values="oos_f1_macro", aggfunc="max"
).round(3)

fig, ax = plt.subplots(figsize=(14, 9))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGn", vmin=0.25, vmax=0.65,
            linewidths=0.3, ax=ax, annot_kws={"size": 8})
ax.set_title("F1-macro OOS — Best result per Ticker × Model", fontsize=12)
plt.tight_layout()
plt.show()

## 5. ML vs DL Comparison

In [ ]:
# category já foi criada na célula de normalização
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
order = ["ML", "CNN", "LSTM", "CNN_LSTM", "Transformer"]
order = [o for o in order if o in df["category"].unique()]
palette = {"ML": "#2ecc71", "CNN": "#95a5a6", "LSTM": "#3498db",
           "CNN_LSTM": "#9b59b6", "Transformer": "#e74c3c", "MLP": "#f39c12"}
for ax, (metric, label) in zip(axes, [("oos_f1_macro", "F1-macro OOS"), ("oos_sharpe", "Sharpe OOS")]):
    if metric not in df.columns:
        ax.set_title(f"{label} — sem dados")
        continue
    sns.boxplot(data=df, x="category", y=metric, order=order,
                palette={k: v for k, v in palette.items() if k in order}, ax=ax)
    ax.set_title(label)
    ax.set_xlabel("")
plt.suptitle("ML vs DL Approaches — OOS Metrics", fontsize=12)
plt.tight_layout()
plt.show()

## 6. Statistical Test — Learned Wavelet vs Baselines

In [ ]:
from src.evaluation import WilcoxonComparison

# Paired Wilcoxon por ticker: learned_wavelet vs raw/db4, por arquitetura
architectures = [m for m in df["model_name"].unique() if m in ["CNN","LSTM","CNN_LSTM","Transformer"]]
results = []
for arch in architectures:
    for baseline in ["raw", "db4"]:
        arch_sub = df[df["model_name"] == arch]
        res_df = WilcoxonComparison.compare_modes(
            results_df=arch_sub,
            metric="oos_f1_macro",
            baseline_mode=baseline,
            comparison_modes=["learned_wavelet"],
            group_col="ticker",
        )
        if not res_df.empty:
            row = res_df.iloc[0].to_dict()
            # Compute means for display
            lw_scores   = arch_sub[arch_sub["mode"] == "learned_wavelet"]["oos_f1_macro"].dropna()
            base_scores = arch_sub[arch_sub["mode"] == baseline]["oos_f1_macro"].dropna()
            row["mean_a"] = float(lw_scores.mean()) if len(lw_scores) > 0 else float("nan")
            row["mean_b"] = float(base_scores.mean()) if len(base_scores) > 0 else float("nan")
            results.append({"arch": arch, "vs": baseline, **row})

if results:
    print(f"{'Arch':<14} {'vs':<6} {'LW mean':>9} {'base mean':>10} {'stat':>7} {'p-value':>9} {'sig':>4}")
    print("-" * 60)
    for r in results:
        sig = "***" if r["p_value"] < 0.01 else ("**" if r["p_value"] < 0.05 else ("*" if r["p_value"] < 0.1 else ""))
        print(f"{r['arch']:<14} {r['vs']:<6} {r['mean_a']:>9.4f} {r['mean_b']:>10.4f} "
              f"{r['statistic']:>7.2f} {r['p_value']:>9.4f} {sig:>4}")
else:
    print("Sem dados suficientes para Wilcoxon.")


## 7. Stability — F1 std across Folds

In [ ]:
if "oos_f1_macro_std" in df.columns:
    stability = (
        df.groupby(["model_name", "mode"])["oos_f1_macro_std"]
        .mean().round(4)
        .sort_values()
    )
    print("F1-macro OOS std (lower = mais estável entre janelas):")
    print(stability.to_string())
else:
    print("Coluna 'oos_f1_macro_std' não encontrada — rode os experimentos primeiro.")